# 第 29 课 · 固定题集与自动评测
## 把“看起来不错”变成可复查的实验

**课程主线**：明确题目与输出约定 → 冻结参考答案 → 测试评分器 → 固定推理设置 → 保存原始回答 → 分层统计 → 配对分析。

本课延续第28课的 `Qwen/Qwen3-0.6B`，但评测工具的离线部分只用 Python 标准库。Notebook **无需同名 `.py` 即可独立使用**。

### 运行边界
- 默认不下载任何文件，不加载真实模型。手工回答夹具明确标为 `HAND_AUTHORED_NOT_A_MODEL`。
- 实际 Qwen 权重推理放在后面的显式开关中；本讲义制作环境没有 Transformers 和模型权重，真实生成单元未执行。
- PyTorch 接口替身测试只测试输入输出封装，不是 Qwen 能力测试。
- 不执行模型生成的代码，不向在线 API 发送题目。不更新权重。

**学习目标**：你能解释评分分母、识别格式错误和运行失败、在相同题集上定位回归，并给每份报告明确的来源与实验条件。

## 0. 从上一课接过来的问题

加载成功、生成成功、格式合规、答案正确、正常结束，是五个不同判断。

本课选择四种有明确答案的任务，而不是让另一个模型自由打分：

| 类别 | 任务 | 输出约定 |
|---|---|---|
| arithmetic | 简单整数计算 | 规范的 ASCII 十进制整数 |
| code_trace | 阅读一小段 Python，预测打印结果 | 一个整数；评分时不执行代码 |
| routing | 按题目给定的字面规则分类 | PAYMENT / DELIVERY / OTHER |
| extraction | 提取名称与数量 | 恰好两个字段的 JSON 对象 |

每类 **1 道开发题 + 6 道固定评测题**，共4+24题。题集是自编教学材料，不是排行榜，不代表完整的推理或编程能力。

> 先在 dev 集排查格式与流程，再冻结设置评测 eval。看过 eval 错题后继续调参，它就成了开发/回归集，不能再声称独立盲测。公开题目也无法证明预训练污染不存在。

In [1]:
from __future__ import annotations

import argparse
import copy
import hashlib
import importlib.metadata
import json
import math
import platform
import re
import statistics
import sys
import tempfile
import time
from collections import Counter
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

MODEL_ID = "Qwen/Qwen3-0.6B"
REVISION = "c1899de289a04d12100db370d81485cdf75e47ca"
FORMAT_VERSION = "lesson29-evaluation-v1"
GRADER_VERSION = "strict-ascii-int-label-json-v1"
TASKS = ("arithmetic", "code_trace", "routing", "extraction")
SYSTEM_PROMPT = "你正在完成固定题集。请严格遵守每道题的输出格式，只输出最终答案，不解释、不使用 Markdown 代码围栏。"
CONTRACTS = {
    "arithmetic": "只输出一个十进制整数。使用 ASCII 数字和必要的负号，不加单位、前导零、加号或解释。",
    "code_trace": "判断下面 Python 3 代码打印的整数。只输出该整数，不执行任何外部操作，不加解释或代码围栏。",
    "routing": "使用字面规则分类：文本含“退款”则输出 PAYMENT；否则含“快递”则输出 DELIVERY；否则输出 OTHER。只输出一个大写标签。",
    "extraction": '只输出一个 JSON 对象，恰好包含 "item" 和 "quantity" 两个字段。item 为名称字符串，quantity 为非负整数。不得增加字段、解释或代码围栏。',
}
SOURCES = {
    "model": f"https://huggingface.co/{MODEL_ID}/blob/{REVISION}/README.md",
    "generation": "https://huggingface.co/docs/transformers/v4.57.3/en/main_classes/text_generation",
    "template": "https://huggingface.co/docs/transformers/chat_templating",
    "json": "https://docs.python.org/3/library/json.html",
    "harness": "https://github.com/EleutherAI/lm-evaluation-harness/blob/main/docs/task_guide.md",
    "reproducibility": "https://docs.pytorch.org/docs/stable/notes/randomness.html",
}


def fingerprint(value: Any) -> str:
    text = json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(",", ":"), allow_nan=False)
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def json_write(path: Path, value: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")


def positive_int(value: Any, name: str) -> int:
    if type(value) is not int or value <= 0:
        raise ValueError(f"{name} 必须是正整数，不接受 bool")
    return value


def environment_info() -> dict:
    versions = {}
    for name in ("torch", "transformers", "tokenizers", "safetensors", "huggingface_hub"):
        try:
            versions[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            versions[name] = None
    return {"python": platform.python_version(), "platform": platform.platform(), "packages": versions}


def unique_run_dir(root: str | Path, prefix: str) -> Path:
    root = Path(root).expanduser()
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
    result = root / f"{prefix}_{stamp}"
    result.mkdir(parents=True, exist_ok=False)
    return result

In [2]:
print(json.dumps(environment_info(), ensure_ascii=False, indent=2))
print("这里只检查环境；没有加载模型。")

{
  "python": "3.13.5",
  "platform": "Linux-6.18.57-x86_64-with-glibc2.41",
  "packages": {
    "torch": "2.10.0+cpu",
    "transformers": null,
    "tokenizers": null,
    "safetensors": "0.7.0",
    "huggingface_hub": "1.16.1"
  }
}
这里只检查环境；没有加载模型。


## 1. 先冻结数据，再定义提示

题目保存 `id、task、split、prompt、expected`。ID 用于配对，不能依赖输出文件的行号。

`public_request()` 只把题干和格式要求交给模型；`expected` 只留在评分侧。

不要把整个题目字典直接拼进用户消息，否则可能把参考答案一起泄漏出去。也不要在模型报错后删除这道题来缩小分母。

正式评测框架同样将题目输入、参考输出、过滤规则与指标配置分开，并保存配置以利于复现。[来源：LM Evaluation Harness](https://github.com/EleutherAI/lm-evaluation-harness/blob/main/docs/task_guide.md)

In [3]:
def make_suite() -> list[dict]:
    """人工编写的可审查题集；不是公开排行榜、秘密测试集或未知污染的保证。"""
    suite = []

    def add(task, split, number, prompt, expected):
        suite.append({"id": f"{split}-{task}-{number:02d}", "task": task,
                      "split": split, "prompt": prompt, "expected": expected})

    add("arithmetic", "dev", 1, "计算：6 + 7。", 13)
    arithmetic = [
        ("计算：17 + 28。", 45),
        ("计算：-7 + 3。", -4),
        ("计算：12 × 13。", 156),
        ("计算：144 ÷ 12。", 12),
        ("计算：3 × (8 + 2) - 7。", 23),
        ("每支笔6元，购买9支，支付100元，应找回多少元？", 46),
    ]
    for i, (prompt, gold) in enumerate(arithmetic, 1):
        add("arithmetic", "eval", i, prompt, gold)
    add("code_trace", "dev", 1, 'x = [1, 3]\nprint(sum(x))', 4)
    traces = [
        ('a = [2, 4, 6]\nprint(sum(a))', 12),
        ('print(len("模型🙂"))', 3),
        ('total = 0\nfor i in range(5):\n    total += i\nprint(total)', 10),
        ('x = [1, 2]\ny = x\ny.append(3)\nprint(len(x))', 3),
        ('d = {"a": 2, "b": 3}\nprint(d.get("c", 4) + d["a"])', 6),
        ('print(sum(i * i for i in range(4)))', 14),
    ]
    for i, (prompt, gold) in enumerate(traces, 1):
        add("code_trace", "eval", i, prompt, gold)
    add("routing", "dev", 1, "请查看快递状态。", "DELIVERY")
    routes = [
        ("我想退款。", "PAYMENT"),
        ("我的快递到哪了？", "DELIVERY"),
        ("谢谢你的帮助。", "OTHER"),
        ("快递损坏，我要求退款。", "PAYMENT"),
        ("设备已修好，无需维修。", "OTHER"),
        ("我不要求退款，只想咨询。", "PAYMENT"),
    ]
    for i, (prompt, gold) in enumerate(routes, 1):
        add("routing", "eval", i, prompt, gold)
    add("extraction", "dev", 1, "名称：白纸；数量：8。", {"item": "白纸", "quantity": 8})
    extracts = [("红茶", 3), ("A-7", 0), ("螺丝", 12), ("短 USB 线", 2),
                ("3号箱", 1), ('"测试"标签', 4)]
    for i, (name, qty) in enumerate(extracts, 1):
        add("extraction", "eval", i, f"名称：{name}；数量：{qty}。", {"item": name, "quantity": qty})
    validate_suite(suite)
    return suite


def valid_expected(task: str, value: Any) -> bool:
    if task in ("arithmetic", "code_trace"):
        return type(value) is int
    if task == "routing":
        return type(value) is str and value in {"PAYMENT", "DELIVERY", "OTHER"}
    if task == "extraction":
        return (type(value) is dict and set(value) == {"item", "quantity"}
                and type(value["item"]) is str and type(value["quantity"]) is int
                and value["quantity"] >= 0)
    return False


def validate_suite(suite: Any) -> None:
    if type(suite) is not list or not suite:
        raise ValueError("题集必须是非空列表")
    ids, texts = set(), set()
    for item in suite:
        if type(item) is not dict or set(item) != {"id", "task", "split", "prompt", "expected"}:
            raise ValueError("题目字段不符合约定")
        if not isinstance(item["id"], str) or not item["id"] or item["id"] in ids:
            raise ValueError("题目 ID 为空或重复")
        if item["task"] not in TASKS or item["split"] not in {"dev", "eval"}:
            raise ValueError("未知任务或 split")
        if not isinstance(item["prompt"], str) or not item["prompt"].strip():
            raise ValueError("题干必须为非空字符串")
        key = (item["task"], item["prompt"])
        if key in texts:
            raise ValueError("跨 split 或同 split 出现重复题干")
        if not valid_expected(item["task"], item["expected"]):
            raise ValueError("参考答案类型或结构错误")
        ids.add(item["id"])
        texts.add(key)


def select_items(suite: list[dict], split="eval", limit: int | None = None) -> list[dict]:
    validate_suite(suite)
    if split not in {"dev", "eval"}:
        raise ValueError("split 必须是 dev/eval")
    selected = [copy.deepcopy(x) for x in suite if x["split"] == split]
    if not selected:
        raise ValueError("这个 split 为空")
    if limit is not None:
        positive_int(limit, "limit")
        selected = selected[:limit]
    return selected


def public_request(item: dict) -> dict:
    """模型后端只接收这份投影：参考答案和评分器内部状态不进入 messages。"""
    return {"id": item["id"], "messages": [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": CONTRACTS[item["task"]] + "\n\n题目：\n" + item["prompt"]},
    ]}


def protocol_for(items: list[dict]) -> dict:
    validate_suite(items)
    core = {"dataset_sha256": fingerprint(items), "ids": [x["id"] for x in items],
            "request_sha256": fingerprint([public_request(x) for x in items]),
            "grader": GRADER_VERSION,
            "primary_rule": "generation_ok AND valid_format AND correct_value AND stop=eos",
            "normalization": "outer whitespace only; JSON key order/whitespace ignored; strict types"}
    return {**core, "protocol_sha256": fingerprint(core)}

In [4]:
suite = make_suite()
dev_items = select_items(suite, "dev")
eval_items = select_items(suite, "eval")
print("开发题：", len(dev_items), "；固定评测题：", len(eval_items))
print("评测任务分布：", dict(Counter(x["task"] for x in eval_items)))
print("题目结构示例：", eval_items[0])
print("模型实际接收：", public_request(eval_items[0]))

开发题： 4 ；固定评测题： 24
评测任务分布： {'arithmetic': 6, 'code_trace': 6, 'routing': 6, 'extraction': 6}
题目结构示例： {'id': 'eval-arithmetic-01', 'task': 'arithmetic', 'split': 'eval', 'prompt': '计算：17 + 28。', 'expected': 45}
模型实际接收： {'id': 'eval-arithmetic-01', 'messages': [{'role': 'system', 'content': '你正在完成固定题集。请严格遵守每道题的输出格式，只输出最终答案，不解释、不使用 Markdown 代码围栏。'}, {'role': 'user', 'content': '只输出一个十进制整数。使用 ASCII 数字和必要的负号，不加单位、前导零、加号或解释。\n\n题目：\n计算：17 + 28。'}]}


### 检查参考答案不会进入提示

下面只改 `expected`，保持题干不变。模型请求应该完全不变；评分协议指纹则必须改变。

哈希用于发现误接文件/改动，不是签名，也不能证明答案一定来自某个真实模型。

In [5]:
altered_item = copy.deepcopy(eval_items[0])
altered_item["expected"] = 999999
print("修改答案后模型请求不变：", public_request(altered_item) == public_request(eval_items[0]))
altered_set = copy.deepcopy(eval_items)
altered_set[0] = altered_item
print("评分协议指纹改变：", protocol_for(altered_set) != protocol_for(eval_items))
print("当前协议指纹：", protocol_for(eval_items)["protocol_sha256"])

修改答案后模型请求不变： True
评分协议指纹改变： True
当前协议指纹： 61697eb46f8d5d1cfb464e37e91eab27cf455123ca4ab4df184dea59de3f5350


## 2. 评分器自身可能有错：从整数反例开始

题目要求“只输出整数45”时，以下输出不能全部视为正确：

| 输出 | 本课判定 |
|---|---|
| `45` 或外围空白后的 `45` | 合规且正确 |
| `145` | 合规但数值错误 |
| `答案是45` | 格式错误，不从解释里提取数字 |
| `45 或 44` | 格式错误，不能因为碰巧包含45就给分 |
| `45.0`、`+45`、`045`、`４５` | 不符合本课事先说明的规范整数格式 |

规则不是“越严越科学”，而是**先明确任务所需格式，后固定执行同一规则**。真实任务可以采用别的解析约定，但修改评分器后必须改变版本并重新评分所有记录。

In [6]:
def strict_json_loads(text: str) -> Any:
    if not isinstance(text, str) or len(text) > 100_000:
        raise ValueError("JSON 必须是长度受限的字符串")

    def pairs_to_dict(pairs):
        result = {}
        for key, value in pairs:
            if key in result:
                raise ValueError(f"JSON 重复键：{key}")
            result[key] = value
        return result

    def reject_constant(value):
        raise ValueError(f"不接受非标准 JSON 常量：{value}")

    return json.loads(text, object_pairs_hook=pairs_to_dict, parse_constant=reject_constant)


def same_typed_value(actual: Any, expected: Any) -> bool:
    """避免 Python 中 True == 1、1.0 == 1 导致宽松评分。"""
    if type(actual) is not type(expected):
        return False
    if type(expected) is dict:
        return (actual.keys() == expected.keys()
                and all(same_typed_value(actual[k], expected[k]) for k in expected))
    if type(expected) is list:
        return len(actual) == len(expected) and all(same_typed_value(a, b) for a, b in zip(actual, expected))
    return actual == expected


def parse_answer(task: str, raw_text: str) -> tuple[Any, str | None]:
    if not isinstance(raw_text, str):
        return None, "output_not_string"
    if len(raw_text) > 100_000:
        return None, "output_too_long"
    text = raw_text.strip()  # 不删除内部空格、标点、大小写、推理过程或代码围栏。
    if not text:
        return None, "empty_output"
    if task in {"arithmetic", "code_trace"}:
        if len(text) > 1000 or re.fullmatch(r"(?:0|-[1-9][0-9]*|[1-9][0-9]*)", text) is None:
            return None, "integer_format"
        return int(text), None
    if task == "routing":
        if text not in {"PAYMENT", "DELIVERY", "OTHER"}:
            return None, "label_format"
        return text, None
    if task == "extraction":
        try:
            value = strict_json_loads(text)
        except (ValueError, RecursionError):
            return None, "json_syntax_or_duplicate_key"
        if not valid_expected(task, value):
            return value, "json_schema"
        return value, None
    raise ValueError("未知任务")


def validate_raw_record(row: Any) -> None:
    if not isinstance(row, dict) or not isinstance(row.get("id"), str):
        raise ValueError("响应必须包含字符串 id")
    if row.get("status") not in {"ok", "error"}:
        raise ValueError("status 必须是 ok/error")
    if row["status"] == "ok":
        if not isinstance(row.get("raw_text"), str):
            raise ValueError("ok 响应缺少 raw_text")
        if row.get("stop_reason") not in {"eos", "length", "other"}:
            raise ValueError("未知停止原因")
    elif not isinstance(row.get("error"), dict):
        raise ValueError("error 响应必须包含错误详情")


def grade_one(item: dict, record: dict | None) -> dict:
    result = {"id": item["id"], "task": item["task"], "present": record is not None,
              "generated": False, "format_valid": False, "answer_correct": False,
              "success": False, "reason": "missing", "parsed": None}
    if record is None:
        return result
    validate_raw_record(record)
    if record["id"] != item["id"]:
        raise ValueError("题目和响应 ID 不一致")
    if record["status"] == "error":
        return {**result, "reason": "generation_error"}
    parsed, error = parse_answer(item["task"], record["raw_text"])
    result.update(generated=True, parsed=parsed, format_valid=error is None)
    if error is not None:
        return {**result, "reason": error}
    correct = same_typed_value(parsed, item["expected"])
    result["answer_correct"] = correct
    if not correct:
        return {**result, "reason": "wrong_value"}
    if record["stop_reason"] != "eos":
        return {**result, "reason": "correct_but_not_eos"}
    return {**result, "success": True, "reason": "success"}

In [7]:
item = eval_items[0]
for text in ["45", " 45\n", "145", "答案是45", "45 或 44", "45.0", "045", "４５"]:
    row = {"id": item["id"], "status": "ok", "raw_text": text, "stop_reason": "eos"}
    result = grade_one(item, row)
    print(repr(text), "→", result["reason"], "；parsed=", result["parsed"])

'45' → success ；parsed= 45
' 45\n' → success ；parsed= 45
'145' → wrong_value ；parsed= 145
'答案是45' → integer_format ；parsed= None
'45 或 44' → integer_format ；parsed= None
'45.0' → integer_format ；parsed= None
'045' → integer_format ；parsed= None
'４５' → integer_format ；parsed= None


## 3. JSON：解析成功不等于字段正确

本课要求恰好包含 `item: str` 和 `quantity: 非负int`。键的顺序和JSON空白不重要，但多余字段、错误类型和错误数值重要。

需要特别处理两点：

1. Python 中 `True == 1`、`1.0 == 1` 都为真，不能只用宽松的 `==` 判断整数类型。
2. Python 的 JSON 解码器默认允许重复键（通常保留最后一个），还默认接受 `NaN` 等扩展。本课主动拒绝这些输入，不默默修复模型回答。

代码围栏和解释也不剥除；重复键通过 `object_pairs_hook` 检测，非标准常量通过 `parse_constant` 拒绝。原始输出有大小限制，避免不受限解析。

[来源：Python JSON 文档](https://docs.python.org/3/library/json.html)

In [8]:
json_item = next(x for x in eval_items if x["id"] == "eval-extraction-01")
json_answers = [
    '{"quantity":3, "item":"红茶"}',
    '{"item":"红茶","quantity":"3"}',
    '{"item":"红茶","quantity":true}',
    '{"item":"红茶","quantity":3.0}',
    '{"item":"错","item":"红茶","quantity":3}',
    '{"item":"红茶","quantity":4}',
    '```json\n{"item":"红茶","quantity":3}\n```',
]
for text in json_answers:
    row = {"id": json_item["id"], "status": "ok", "raw_text": text, "stop_reason": "eos"}
    print(text, "→", grade_one(json_item, row)["reason"])
print("Python相等性陷阱：", True == 1, 1.0 == 1)

{"quantity":3, "item":"红茶"} → success
{"item":"红茶","quantity":"3"} → json_schema
{"item":"红茶","quantity":true} → json_schema
{"item":"红茶","quantity":3.0} → json_schema
{"item":"错","item":"红茶","quantity":3} → json_syntax_or_duplicate_key
{"item":"红茶","quantity":4} → wrong_value
```json
{"item":"红茶","quantity":3}
``` → json_syntax_or_duplicate_key
Python相等性陷阱： True True


## 4. 一个主指标不够：分开统计四个层面

本课所有主要比例的分母都是**计划评测的题目数 N**：

- 生成覆盖率：成功得到文本的题目 / N。
- 格式通过率：满足输出约定的题目 / N。
- 答案正确率：解析后值正确的题目 / N，不要求EOS。
- **严格成功率（主指标）**：生成成功、格式正确、值正确、正常EOS结束的题目 / N。

“正确却用尽生成预算”会进入答案正确计数，但不进入严格成功。本课采取保守的任务完成约定；这不是所有公开基准都采用的规则。EOS本身也不能证明答案正确。

每条记录同时保留停止原因。`missing`、`generation_error`、`json_schema`、`wrong_value`等分类帮助定位问题，不把基础设施错误误称为推理能力差。

In [9]:
def summary_metrics(graded: list[dict], records: list[dict]) -> dict:
    if not graded:
        raise ValueError("不能汇总空题集")
    n = len(graded)
    by_task = {}
    for task in sorted({x["task"] for x in graded}):
        subset = [x for x in graded if x["task"] == task]
        by_task[task] = {"n": len(subset), "success": sum(x["success"] for x in subset),
                         "success_rate": sum(x["success"] for x in subset) / len(subset)}
    ok = [x for x in records if x["status"] == "ok"]
    durations = [x["generate_seconds"] for x in ok
                 if type(x.get("generate_seconds")) in (int, float)
                 and math.isfinite(x["generate_seconds"]) and x["generate_seconds"] >= 0]
    token_counts = [x["generated_tokens"] for x in ok if type(x.get("generated_tokens")) is int]
    result = {"n": n, "received": sum(x["present"] for x in graded),
              "generated": sum(x["generated"] for x in graded),
              "format_valid": sum(x["format_valid"] for x in graded),
              "answer_correct": sum(x["answer_correct"] for x in graded),
              "success": sum(x["success"] for x in graded),
              "micro_success_rate": sum(x["success"] for x in graded) / n,
              "macro_success_rate": statistics.mean(x["success_rate"] for x in by_task.values()),
              "generation_coverage": sum(x["generated"] for x in graded) / n,
              "answer_accuracy": sum(x["answer_correct"] for x in graded) / n,
              "format_rate_all_items": sum(x["format_valid"] for x in graded) / n,
              "per_task": by_task, "failure_counts": dict(Counter(x["reason"] for x in graded)),
              "stop_counts_generated_only": dict(Counter(x["stop_reason"] for x in ok)),
              "median_generate_seconds": statistics.median(durations) if durations else None,
              "mean_generated_tokens": statistics.mean(token_counts) if token_counts else None,
              "timing_samples": len(durations),
              "timing_scope": "single-request generate, prefill+decode; no load/tokenization; not TTFT or controlled benchmark"}
    return result


def score_records(items: list[dict], records: list[dict]) -> dict:
    validate_suite(items)
    expected_ids = {x["id"] for x in items}
    indexed = {}
    for row in records:
        validate_raw_record(row)
        if row["id"] not in expected_ids:
            raise ValueError("响应包含题集之外的 ID")
        if row["id"] in indexed:
            raise ValueError("重复响应 ID；禁止重试后挑选最好的回答")
        indexed[row["id"]] = row
    graded = [grade_one(item, indexed.get(item["id"])) for item in items]
    return {"graded": graded, "metrics": summary_metrics(graded, records)}


def build_report(items: list[dict], records: list[dict], *, kind: str,
                 condition: dict, scope: str, complete: bool = True) -> dict:
    if kind not in {"hand_authored_fixture", "real_model"}:
        raise ValueError("必须明确是真实模型还是手工夹具")
    if scope not in {"dev", "full_eval", "smoke_subset"}:
        raise ValueError("未知评测范围")
    scored = score_records(items, records)
    return {"format": FORMAT_VERSION, "kind": kind, "scope": scope, "complete": complete,
            "protocol": protocol_for(items), "condition": copy.deepcopy(condition),
            "items": copy.deepcopy(items), "raw_records": copy.deepcopy(records),
            "raw_records_sha256": fingerprint(records), **scored,
            "limitations": ["人工教学题集，不是通用能力基准；不能证明训练污染不存在。",
                            "严格成功率要求正确且正常EOS结束，是本课协议，不是所有基准的通用规则。",
                            "看过评测答案后调参，题集就成为开发/回归集；不可再宣称独立盲测。"]}


def verify_report(report: dict) -> dict:
    if report.get("format") != FORMAT_VERSION:
        raise ValueError("报告格式不匹配")
    if report.get("protocol") != protocol_for(report["items"]):
        raise ValueError("题集/提示/评分协议指纹不匹配")
    if report.get("raw_records_sha256") != fingerprint(report["raw_records"]):
        raise ValueError("原始响应指纹不匹配")
    fresh = score_records(report["items"], report["raw_records"])
    if fresh["graded"] != report.get("graded") or fresh["metrics"] != report.get("metrics"):
        raise ValueError("保存的评分与原始响应不一致；必须重新评分")
    return fresh


def paired_compare(before: dict, after: dict) -> dict:
    verify_report(before)
    verify_report(after)
    if not before["complete"] or not after["complete"]:
        raise ValueError("未完成的运行不能宣称完整对比")
    if before["kind"] != after["kind"]:
        raise ValueError("不能把手工夹具与真实模型成绩比较")
    if before["protocol"] != after["protocol"] or before["scope"] != after["scope"]:
        raise ValueError("配对比较要求相同题集、提示、评分器、范围与题目ID顺序")
    a = {x["id"]: x["success"] for x in before["graded"]}
    b = {x["id"]: x["success"] for x in after["graded"]}
    fixed = [k for k in a if not a[k] and b[k]]
    regressed = [k for k in a if a[k] and not b[k]]
    changed = {k: {"before": before["condition"].get(k), "after": after["condition"].get(k)}
               for k in sorted(before["condition"].keys() | after["condition"].keys())
               if before["condition"].get(k) != after["condition"].get(k)}
    return {"n": len(a), "before_success": sum(a.values()), "after_success": sum(b.values()),
            "fixed_ids": fixed, "regressed_ids": regressed,
            "both_correct": sum(a[k] and b[k] for k in a),
            "both_incorrect": sum(not a[k] and not b[k] for k in a),
            "delta_percentage_points": 100 * (sum(b.values()) - sum(a.values())) / len(a),
            "changed_conditions": changed,
            "note": "配对描述而非显著性检验；多项条件改变时不得归因于单一改动。"}

In [10]:
base_row = {"id": eval_items[0]["id"], "status": "ok", "raw_text": "45"}
for stopping in ("eos", "length", "other"):
    graded = grade_one(eval_items[0], {**base_row, "stop_reason": stopping})
    print(stopping, "答案正确=", graded["answer_correct"], "严格成功=", graded["success"])
print("完全没有返回记录时：", score_records(eval_items, [])["metrics"]["micro_success_rate"])
print("分母仍是：", score_records(eval_items, [])["metrics"]["n"])

eos 答案正确= True 严格成功= True
length 答案正确= True 严格成功= False
other 答案正确= True 严格成功= False
完全没有返回记录时： 0.0
分母仍是： 24


## 5. Micro 与 Macro 不是同一件事

Micro（按题目加权）= 全部成功题数 / 全部题数。

Macro（按任务等权）= 各任务成功率的平均值。

如果任务A有9道题全对，任务B只有1道题且答错：

$$\mathrm{micro}=9/10=90\%,\qquad\mathrm{macro}=(100\%+0\%)/2=50\%.$$

本课四类各6题，所以完整题集的两项指标相同。但将来题数不平衡，不能只看整体数字。只跑一部分题的 smoke run 也不能包装成完整评测。

In [11]:
print("不平衡例子 micro:", 9 / 10)
print("不平衡例子 macro:", (1 + 0) / 2)
print("24题里一题变化的百分点：", 100 / 24)

不平衡例子 micro: 0.9
不平衡例子 macro: 0.5
24题里一题变化的百分点： 4.166666666666667


## 6. 用手工故障夹具验证全流程（不是模型成绩）

接下来人为构造正确答案、格式错误、错误数值、预算停止、运行报错和缺失记录。

这些文本来自课程作者，不是Qwen、不是API返回、也不是伪装成模型的推理实现。夹具允许查看参考答案，因为它的目的只是验证评分程序。

**把模型回答函数与评分函数拆开，才能在没有GPU时调试评测工程。** 这不是把夹具结果当作能力评测的理由。

In [12]:
def fixture_records(items: list[dict]) -> list[dict]:
    rows = []
    for item in items:
        task = item["task"]
        i = int(item["id"].rsplit("-", 1)[1])
        gold = item["expected"]
        text = json.dumps(gold, ensure_ascii=False) if task == "extraction" else str(gold)
        row = {"id": item["id"], "status": "ok", "raw_text": text, "stop_reason": "eos",
               "origin": "HAND_AUTHORED_NOT_A_MODEL", "generated_tokens": None,
               "generate_seconds": None}
        if task == "arithmetic" and i == 5:
            row["raw_text"] = "答案是23"
        elif task == "arithmetic" and i == 6:
            row["stop_reason"] = "length"
        elif task == "extraction" and i == 1:
            row["raw_text"] = '{"quantity":3, "item":"红茶"}'
        elif task == "extraction" and i == 4:
            row["raw_text"] = '{"item":"短 USB 线","quantity":"2"}'
        elif task == "extraction" and i == 5:
            row["raw_text"] = '{"item":"错","item":"3号箱","quantity":1}'
        elif task == "extraction" and i == 6:
            row["raw_text"] = '{"item":"错误名称","quantity":4}'
        elif task == "routing" and i == 6:
            row = {"id": item["id"], "status": "error", "raw_text": None,
                   "error": {"type": "SimulatedFailure", "message": "手工构造的运行错误，不是真实CUDA错误"},
                   "origin": "HAND_AUTHORED_NOT_A_MODEL"}
        elif task == "code_trace" and i == 4:
            row["raw_text"] = "3\n因为列表共享。"
        elif task == "code_trace" and i == 5:
            continue  # 刻意缺失一题，验证总分母不缩小。
        elif task == "code_trace" and i == 6:
            row["raw_text"] = "16"
        rows.append(row)
    return rows


def offline_demo(output_dir="lesson_29_outputs") -> dict:
    items = select_items(make_suite(), "eval")
    rows = fixture_records(items)
    before = build_report(items, rows, kind="hand_authored_fixture",
                          condition={"fixture": "A"}, scope="full_eval")
    changed = copy.deepcopy(rows)
    for row in changed:
        if row["id"] == "eval-arithmetic-05":
            row["raw_text"] = "23"  # 修好一题。
        if row["id"] == "eval-code_trace-01":
            row["raw_text"] = "999"  # 同时退步一题，总分不变。
    after = build_report(items, changed, kind="hand_authored_fixture",
                         condition={"fixture": "B"}, scope="full_eval")
    comparison = paired_compare(before, after)
    directory = unique_run_dir(output_dir, "fixture_NOT_MODEL")
    json_write(directory / "fixture_a.json", before)
    json_write(directory / "fixture_b.json", after)
    json_write(directory / "comparison.json", comparison)
    print("手工夹具演示（绝非Qwen评测结果）：")
    print(json.dumps(before["metrics"], ensure_ascii=False, indent=2))
    print("配对变化：", json.dumps(comparison, ensure_ascii=False))
    print("报告目录：", directory)
    return {"run_dir": str(directory), "before": before, "after": after, "comparison": comparison}

In [13]:
fixture_rows = fixture_records(eval_items)
fixture_a = build_report(eval_items, fixture_rows, kind="hand_authored_fixture",
                         condition={"fixture":"A"}, scope="full_eval")
print("【手工夹具结果，绝非Qwen成绩】")
print(json.dumps(fixture_a["metrics"], ensure_ascii=False, indent=2))

【手工夹具结果，绝非Qwen成绩】
{
  "n": 24,
  "received": 23,
  "generated": 22,
  "format_valid": 18,
  "answer_correct": 16,
  "success": 15,
  "micro_success_rate": 0.625,
  "macro_success_rate": 0.625,
  "generation_coverage": 0.9166666666666666,
  "answer_accuracy": 0.6666666666666666,
  "format_rate_all_items": 0.75,
  "per_task": {
    "arithmetic": {
      "n": 6,
      "success": 4,
      "success_rate": 0.6666666666666666
    },
    "code_trace": {
      "n": 6,
      "success": 3,
      "success_rate": 0.5
    },
    "extraction": {
      "n": 6,
      "success": 3,
      "success_rate": 0.5
    },
    "routing": {
      "n": 6,
      "success": 5,
      "success_rate": 0.8333333333333334
    }
  },
  "failure_counts": {
    "success": 15,
    "integer_format": 2,
    "correct_but_not_eos": 1,
    "missing": 1,
    "wrong_value": 2,
    "generation_error": 1,
    "json_schema": 1,
    "json_syntax_or_duplicate_key": 1
  },
  "stop_counts_generated_only": {
    "eos": 21,
    "length": 1


### 对照你看到的数字

计划24题，收到23条记录，其中1条是运行错误；因此成功得到文本的是22题。

有效格式18题，值正确16题，其中1题没有正常EOS结束。严格成功因此是 **15/24=62.5%**，不是15/22，也不是15/23。

时间与生成token数为 `null`，因为这些是手工文本，不能捏造推理速度或token统计。

In [14]:
assert fixture_a["metrics"]["n"] == 24
assert fixture_a["metrics"]["received"] == 23
assert fixture_a["metrics"]["generated"] == 22
assert fixture_a["metrics"]["answer_correct"] == 16
assert fixture_a["metrics"]["success"] == 15
for row in fixture_a["graded"]:
    if not row["success"]:
        print(row["id"], "→", row["reason"])

eval-arithmetic-05 → integer_format
eval-arithmetic-06 → correct_but_not_eos
eval-code_trace-04 → integer_format
eval-code_trace-05 → missing
eval-code_trace-06 → wrong_value
eval-routing-06 → generation_error
eval-extraction-04 → json_schema
eval-extraction-05 → json_syntax_or_duplicate_key
eval-extraction-06 → wrong_value


## 7. 总分不变，也可能修好一题又弄坏一题

比较两个实验，先要求题集、ID、提示和评分协议相同，然后逐题配对。

四类结果：两边都对、两边都错、修好的题、退步的题。不能把两份记录简单按当前行号 `zip()`，也不能只比较总百分比。

报告会列出实验条件差异。如果同时改变模型、精度、生成预算和模板，不能把分数变化都归因于量化或某个单一设置。

In [15]:
fixture_b_rows = copy.deepcopy(fixture_rows)
for row in fixture_b_rows:
    if row["id"] == "eval-arithmetic-05":
        row["raw_text"] = "23"
    if row["id"] == "eval-code_trace-01":
        row["raw_text"] = "999"
fixture_b = build_report(eval_items, fixture_b_rows, kind="hand_authored_fixture",
                         condition={"fixture":"B"}, scope="full_eval")
paired = paired_compare(fixture_a, fixture_b)
print(json.dumps(paired, ensure_ascii=False, indent=2))

{
  "n": 24,
  "before_success": 15,
  "after_success": 15,
  "fixed_ids": [
    "eval-arithmetic-05"
  ],
  "regressed_ids": [
    "eval-code_trace-01"
  ],
  "both_correct": 14,
  "both_incorrect": 8,
  "delta_percentage_points": 0.0,
  "changed_conditions": {
    "fixture": {
      "before": "A",
      "after": "B"
    }
  },
  "note": "配对描述而非显著性检验；多项条件改变时不得归因于单一改动。"
}


### 小样本不应该过度解释

本课只有24道人工题，一题就是约4.17个百分点。改对一题不证明整体能力有统计上可靠的提升。

不把同一道题用不同种子生成的答案当作独立题目来增加分母，不在多个随机回答中挑最好的一次再称作一次采样正确率。本课每题恰好生成一次，禁止同ID重复响应。

训练集污染、题型覆盖与分数不确定性需要更大、更独立的评测设计；本课首先建立可审计的回归基线。

## 8. 真实生成配置必须写进报告

默认沿用第28课固定仓库commit的 `Qwen/Qwen3-0.6B`，关闭thinking、batch=1、最多64个新增token。

默认使用贪心以便建立受控基线，**不是宣称这是官方推荐的最佳质量配置**。Qwen模型卡为非thinking模式提供了采样建议；本课的 `--decode sample` 明确使用 temperature=0.7、top_p=0.8、top_k=20。

[模型卡](https://huggingface.co/Qwen/Qwen3-0.6B/blob/c1899de289a04d12100db370d81485cdf75e47ca/README.md)

从新的 `GenerationConfig` 构建本课设置，避免仓库默认的采样/重复惩罚设置悄悄参与。EOS列表从配套配置取得，不能漏掉第二个停止ID。

[GenerationConfig 文档](https://huggingface.co/docs/transformers/v4.57.3/en/main_classes/text_generation)

In [16]:
@dataclass
class RunSettings:
    model: str = MODEL_ID
    revision: str = REVISION
    device: str = "cpu"
    dtype: str = "auto"
    split: str = "dev"
    limit: int | None = None
    max_input: int = 512
    max_new: int = 64
    context_limit: int = 2048
    decode: str = "greedy"
    seed: int = 29
    local_files_only: bool = False
    output_dir: str = "lesson_29_outputs"


def stable_item_seed(base: int, item_id: str) -> int:
    if type(base) is not int or base < 0:
        raise ValueError("seed 必须是非负整数")
    # 不使用 Python hash()，其进程间结果可能改变。
    return int(fingerprint({"seed": base, "id": item_id})[:8], 16) % (2**31)


def decode_options(settings: RunSettings, stop_ids: list[int], pad_id: int) -> dict:
    positive_int(settings.max_new, "max_new")
    if settings.decode not in {"greedy", "sample"}:
        raise ValueError("decode 必须是 greedy/sample")
    if not stop_ids or any(type(x) is not int or x < 0 for x in stop_ids):
        raise ValueError("EOS ID 列表不合法")
    if type(pad_id) is not int or pad_id < 0:
        raise ValueError("PAD ID 不合法")
    cfg = {"max_new_tokens": settings.max_new, "do_sample": settings.decode == "sample",
           "num_beams": 1, "num_return_sequences": 1, "use_cache": True,
           "eos_token_id": list(dict.fromkeys(stop_ids)), "pad_token_id": pad_id,
           "return_dict_in_generate": True, "output_scores": False}
    if settings.decode == "sample":
        cfg.update(temperature=0.7, top_p=0.8, top_k=20)
    return cfg


def split_generated_ids(sequence: list[int], prefix: list[int], eos: list[int], budget: int) -> dict:
    positive_int(budget, "budget")
    if sequence[:len(prefix)] != prefix or len(sequence) < len(prefix):
        raise ValueError("生成结果没有保留原输入前缀")
    new = sequence[len(prefix):]
    if len(new) > budget:
        raise ValueError("生成长度超过设定预算")
    positions = [i for i, token in enumerate(new) if token in eos]
    if positions:
        first = positions[0]
        return {"visible_ids": new[:first], "generated_ids": new[:first + 1], "stop_reason": "eos"}
    return {"visible_ids": new, "generated_ids": new,
            "stop_reason": "length" if len(new) == budget else "other"}


def require_real_dependencies():
    try:
        import torch
        import transformers
        from transformers import AutoConfig, AutoTokenizer, AutoModelForCausalLM, GenerationConfig
    except ImportError as exc:
        raise RuntimeError("真实路径需要现有PyTorch环境，以及 transformers>=4.57,<6 和 safetensors。离线评分不需要它们。") from exc
    nums = tuple(int(x) for x in re.findall(r"\d+", transformers.__version__)[:2])
    if not (4, 57) <= nums < (6, 0):
        raise RuntimeError("本课接口范围为 transformers>=4.57,<6；请使用独立环境或沿用第28课环境。")
    return torch, AutoConfig, AutoTokenizer, AutoModelForCausalLM, GenerationConfig


def choose_device(torch, device: str, dtype: str):
    if device == "auto":
        device = "cuda" if torch.cuda.is_available() else "cpu"
    if device not in {"cpu", "cuda"}:
        raise ValueError("仅支持 cpu/cuda/auto 单设备")
    if device == "cuda" and not torch.cuda.is_available():
        raise RuntimeError("CUDA不可用，不自动回退到CPU")
    if dtype == "auto":
        dtype = "float16" if device == "cuda" else "float32"
    if dtype not in {"float32", "float16", "bfloat16"}:
        raise ValueError("未知dtype")
    if device == "cpu" and dtype == "float16":
        raise ValueError("本课不在CPU使用FP16；使用FP32")
    if device == "cuda" and dtype == "bfloat16" and not torch.cuda.is_bf16_supported():
        raise RuntimeError("当前CUDA环境未报告BF16支持")
    return torch.device(device), getattr(torch, dtype)


def inspect_source(settings: RunSettings, items: list[dict]) -> tuple:
    torch, AutoConfig, AutoTokenizer, AutoModel, GenerationConfig = require_real_dependencies()
    source_path = Path(settings.model).expanduser()
    if source_path.is_dir():
        source = str(source_path.resolve())
        options = {"local_files_only": True, "trust_remote_code": False}
    else:
        if settings.model != MODEL_ID or re.fullmatch(r"[0-9a-f]{40}", settings.revision) is None:
            raise ValueError("远程来源限固定commit的Qwen/Qwen3-0.6B；也可用本地Qwen3目录")
        source = settings.model
        options = {"revision": settings.revision, "local_files_only": settings.local_files_only,
                   "trust_remote_code": False}
    config = AutoConfig.from_pretrained(source, **options)
    if config.model_type != "qwen3":
        raise ValueError("本课真实接口仅支持Qwen3，不猜测其他模型的模板")
    tokenizer = AutoTokenizer.from_pretrained(source, padding_side="left", **options)
    if not isinstance(tokenizer.chat_template, str) or "enable_thinking" not in tokenizer.chat_template:
        raise ValueError("缺少配套Qwen3模板或thinking控制")
    if tokenizer.pad_token_id is None:
        raise ValueError("配套tokenizer没有PAD ID")
    maximum = max(tokenizer.get_vocab().values())
    if maximum >= config.vocab_size:
        raise ValueError("分词器ID超过配置中的词表大小")
    inspected = []
    for item in items:
        request = public_request(item)
        rendered = tokenizer.apply_chat_template(request["messages"], tokenize=False,
                         add_generation_prompt=True, enable_thinking=False)
        ids = tokenizer(rendered, add_special_tokens=False)["input_ids"]
        direct = tokenizer.apply_chat_template(request["messages"], tokenize=True,
                         add_generation_prompt=True, enable_thinking=False)
        if ids != direct:
            raise RuntimeError("模板两种编码路线不一致")
        if len(ids) > settings.max_input or len(ids) + settings.max_new > min(settings.context_limit, config.max_position_embeddings):
            raise ValueError(f"{item['id']} 超过预算，不静默截断")
        inspected.append({"id": item["id"], "messages": request["messages"],
                          "rendered_prompt": rendered, "input_ids": ids, "input_tokens": len(ids)})
    report = {"source": source, "requested_revision": options.get("revision"),
              "resolved_config_commit": getattr(config, "_commit_hash", None),
              "config_sha256": fingerprint(config.to_dict()),
              "template_sha256": fingerprint(tokenizer.chat_template),
              "tokenizer_sha256": fingerprint(tokenizer.backend_tokenizer.to_str()),
              "prompts": inspected,
              "weights_loaded": False, "enable_thinking": False}
    return torch, AutoModel, GenerationConfig, config, tokenizer, report, source, options

In [17]:
settings_example = RunSettings(split="eval", decode="greedy")
print("默认明确设置：", decode_options(settings_example, [151645,151643], 151643))
print("按题目ID派生的固定seed：", stable_item_seed(29, "eval-arithmetic-01"))
print("修改题目遍历顺序不会改变同ID的seed。")

默认明确设置： {'max_new_tokens': 64, 'do_sample': False, 'num_beams': 1, 'num_return_sequences': 1, 'use_cache': True, 'eos_token_id': [151645, 151643], 'pad_token_id': 151643, 'return_dict_in_generate': True, 'output_scores': False}
按题目ID派生的固定seed： 1143175085
修改题目遍历顺序不会改变同ID的seed。


## 9. 生成与评分分离，保留原始证据

真实生成函数只接收公共请求，不接收 `expected`。模板文本、输入ID、生成ID、原始输出、停止原因和实际生效配置都保存。

只切掉完整输入前缀，再移除明确的EOS尾部；不自动剥离围栏、解释或其他控制标记替模型修复格式。模板先渲染后编码时，明确 `add_special_tokens=False`。

[Chat Template 文档](https://huggingface.co/docs/transformers/chat_templating)

每题独立，前一题的回答不会偷偷进入下一题上下文。`eval()`和`inference_mode()`控制推理模式；随机状态在调用后恢复。但固定seed不保证跨库版本、跨硬件逐位相同。

[PyTorch 可复现性](https://docs.pytorch.org/docs/stable/notes/randomness.html)

In [18]:
def generate_one(model, tokenizer, request: dict, settings: RunSettings,
                 device, generation_factory) -> dict:
    import torch
    if set(request) != {"id", "messages"}:
        raise ValueError("模型后端只接受id/messages，禁止传入expected")
    rendered = tokenizer.apply_chat_template(request["messages"], tokenize=False,
                         add_generation_prompt=True, enable_thinking=False)
    inputs = tokenizer(rendered, return_tensors="pt", add_special_tokens=False,
                       truncation=False, return_attention_mask=True)
    if inputs["input_ids"].ndim != 2 or inputs["input_ids"].shape[0] != 1:
        raise ValueError("本课基线固定batch=1")
    width = inputs["input_ids"].shape[1]
    if (width > settings.max_input or width + settings.max_new >
            min(settings.context_limit, model.config.max_position_embeddings)):
        raise ValueError("输入与输出预算超出约定，不静默截断")
    inputs = {k: v.to(device) for k, v in inputs.items()}
    ids = getattr(model.generation_config, "eos_token_id", None)
    if ids is None:
        ids = tokenizer.eos_token_id
    stop_ids = [ids] if type(ids) is int else list(ids)
    cfg = decode_options(settings, stop_ids, tokenizer.pad_token_id)
    generation_config = generation_factory(**cfg)  # 不继承仓库中隐藏的采样/重复惩罚默认值。
    effective_seed = stable_item_seed(settings.seed, request["id"])
    modes = [(module, module.training) for module in model.modules()]
    devices = list(range(torch.cuda.device_count())) if torch.cuda.is_available() else []
    try:
        model.eval()
        with torch.random.fork_rng(devices=devices), torch.inference_mode():
            torch.manual_seed(effective_seed)
            if device.type == "cuda":
                torch.cuda.synchronize(device)
            start = time.perf_counter()
            output = model.generate(**inputs, generation_config=generation_config)
            if device.type == "cuda":
                torch.cuda.synchronize(device)
            elapsed = time.perf_counter() - start
        prefix = inputs["input_ids"][0].cpu().tolist()
        generated = split_generated_ids(output.sequences[0].cpu().tolist(), prefix,
                                         stop_ids, settings.max_new)
        # 只移除明确的EOS尾部，不把其他特殊控制符悄悄删掉来替模型修复格式。
        raw_text = tokenizer.decode(generated["visible_ids"], skip_special_tokens=False,
                                     clean_up_tokenization_spaces=False)
        decoded_all = tokenizer.decode(generated["generated_ids"], skip_special_tokens=False,
                                        clean_up_tokenization_spaces=False)
    finally:
        for module, mode in modes:
            module.training = mode
    return {"id": request["id"], "status": "ok", "raw_text": raw_text,
            "raw_with_eos": decoded_all, "stop_reason": generated["stop_reason"],
            "input_tokens": width, "generated_tokens": len(generated["generated_ids"]),
            "generated_token_ids": generated["generated_ids"], "input_ids": prefix,
            "messages": request["messages"], "rendered_prompt": rendered,
            "prompt_sha256": fingerprint(request["messages"]), "effective_seed": effective_seed,
            "generation_config": generation_config.to_dict(), "generate_seconds": elapsed}


def local_source_hashes(source: str) -> dict | None:
    path = Path(source)
    if not path.is_dir():
        return None
    hashes = {}
    for file in sorted(path.iterdir()):
        if file.is_file() and file.suffix in {".json", ".safetensors", ".bin", ".model"}:
            digest = hashlib.sha256()
            with file.open("rb") as stream:
                for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b""):
                    digest.update(chunk)
            hashes[file.name] = {"bytes": file.stat().st_size, "sha256": digest.hexdigest()}
    return hashes


def run_real(settings: RunSettings, *, inspect_only=False) -> dict:
    for field in ("max_input", "max_new", "context_limit"):
        positive_int(getattr(settings, field), field)
    items = select_items(make_suite(), settings.split, settings.limit)
    scope = "smoke_subset" if settings.limit is not None and len(items) < len(select_items(make_suite(), settings.split)) else (
        "dev" if settings.split == "dev" else "full_eval")
    output = Path(settings.output_dir).expanduser().resolve()
    model_path = Path(settings.model).expanduser()
    if model_path.is_dir() and output.is_relative_to(model_path.resolve()):
        raise ValueError("输出目录不能位于输入模型目录内")
    directory = unique_run_dir(output, "inspect" if inspect_only else "real")
    manifest = {"format": FORMAT_VERSION, "status": "started", "kind": "real_model",
                "settings": asdict(settings), "scope": scope, "protocol": protocol_for(items),
                "environment": environment_info(), "sources": SOURCES,
                "privacy": "保存输入输出明文，不要放入密码或私人数据"}
    json_write(directory / "manifest.json", manifest)
    json_write(directory / "dataset.json", items)
    json_write(directory / "requests.json", [public_request(x) for x in items])
    try:
        torch, AutoModel, GenConfig, config, tokenizer, inspection, source, options = inspect_source(settings, items)
        json_write(directory / "inspection.json", inspection)
        if inspect_only:
            manifest.update(status="inspection_only", weights_loaded=False)
            json_write(directory / "manifest.json", manifest)
            print("只检查了分词器，未加载权重、未评分：", directory)
            return {"run_dir": str(directory), "manifest": manifest}
        device, dtype = choose_device(torch, settings.device, settings.dtype)
        model = AutoModel.from_pretrained(source, dtype=dtype, use_safetensors=True,
                                          attn_implementation="sdpa", **options).to(device)
        model.eval()
        if max(tokenizer.get_vocab().values()) >= model.get_input_embeddings().num_embeddings:
            raise ValueError("tokenizer与实际Embedding范围不匹配")
        device_name = torch.cuda.get_device_name(device) if device.type == "cuda" else platform.processor()
        local_hashes = local_source_hashes(source)
        condition = {"model": source, "revision": options.get("revision"),
                     "local_file_hashes": local_hashes, "dtype": str(dtype), "device": str(device),
                     "device_name": device_name, "torch_threads": torch.get_num_threads(),
                     "packages": environment_info()["packages"], "decode": settings.decode,
                     "max_new_tokens": settings.max_new, "max_input": settings.max_input,
                     "context_limit": settings.context_limit, "seed": settings.seed,
                     "enable_thinking": False, "batch_size": 1,
                     "attention_requested": "sdpa_not_a_kernel_guarantee",
                     "template_sha256": inspection["template_sha256"],
                     "tokenizer_sha256": inspection["tokenizer_sha256"],
                     "config_sha256": inspection["config_sha256"]}
        manifest.update(status="generating", condition=condition, weights_loaded=True)
        json_write(directory / "manifest.json", manifest)
        rows, interrupted = [], False
        with (directory / "raw.jsonl").open("w", encoding="utf-8") as stream:
            for index, item in enumerate(items, 1):
                try:
                    row = generate_one(model, tokenizer, public_request(item), settings, device, GenConfig)
                except Exception as exc:
                    row = {"id": item["id"], "status": "error", "raw_text": None,
                           "error": {"type": type(exc).__name__, "message": str(exc)}}
                    interrupted = True
                rows.append(row)
                stream.write(json.dumps(row, ensure_ascii=False, allow_nan=False) + "\n")
                stream.flush()
                print(f"[{index}/{len(items)}] {item['id']}: {grade_one(item, row)['reason']}")
                if interrupted:
                    # 单设备故障后不自动改参数/重试。其余计划题目保持缺失并计入分母。
                    break
        report = build_report(items, rows, kind="real_model", condition=condition,
                              scope=scope, complete=not interrupted)
        json_write(directory / "report.json", report)
        manifest.update(status="generation_error" if interrupted else "completed",
                        received=len(rows), planned=len(items), raw_records_sha256=fingerprint(rows))
        json_write(directory / "manifest.json", manifest)
        print(json.dumps(report["metrics"], ensure_ascii=False, indent=2))
        print("真实评测目录：", directory)
        return {"run_dir": str(directory), "manifest": manifest, "report": report}
    except Exception as exc:
        manifest.update(status="failed_before_complete_run", error_type=type(exc).__name__, error=str(exc))
        json_write(directory / "manifest.json", manifest)
        print("失败记录：", directory / "manifest.json", file=sys.stderr)
        raise

### 错误处理与时间口径

单题生成错误记录为 `error`，此后停止本次真实运行，未做的题仍保留在计划分母中。不会自动降低精度/缩短预算/更换设备重试后，只保留成功结果。未完成运行禁止做完整配对比较。

`generate_seconds` 是单请求 `generate()` 的墙钟时间，包括prefill与decode，不包括下载、模型加载和分词。CUDA计时边界同步。它是诊断数据，不是稳定吞吐基准、首token延迟或整机服务响应时间。

本课不计算把不同回答长度混在一起的“质量加速比”。要比较纯速度，回到第23课的固定工作量基准。

## 10. 运行实际模型前，先 inspect

沿用能运行第28课的Python环境。缺少依赖时，在那个环境的终端安装：

```bash
python -m pip install "transformers>=4.57,<6" safetensors
```

先开 `RUN_REAL_INSPECTION` 检查4道开发题的配置、模板和token预算，不加载权重。没有完整缓存且离线时会明确失败，不改用手工夹具冒充真实输出。

开发题检查通过后，开启生成。`--device cuda` 不可用会报错；CPU使用FP32。首次需要下载第28课同一模型权重，已有匹配缓存可复用。

In [19]:
RUN_REAL_INSPECTION = False
RUN_REAL_EVALUATION = False
REAL_DEVICE = "cpu"          # 3060已配置好后改为 "cuda"
REAL_DTYPE = "auto"          # cpu→FP32；cuda→FP16
REAL_SPLIT = "dev"           # 先4题排错；冻结设置后改 "eval" 跑24题
LOCAL_FILES_ONLY = False     # 依赖与所需缓存已完整时可改True
REAL_OUTPUT_DIR = "lesson_29_outputs"

real_settings = RunSettings(device=REAL_DEVICE, dtype=REAL_DTYPE, split=REAL_SPLIT,
                            local_files_only=LOCAL_FILES_ONLY, output_dir=REAL_OUTPUT_DIR)
if RUN_REAL_INSPECTION:
    inspected_run = run_real(real_settings, inspect_only=True)
else:
    print("真实分词器检查：未开启；没有下载，没有伪造检查结果。")

真实分词器检查：未开启；没有下载，没有伪造检查结果。


In [20]:
if RUN_REAL_EVALUATION:
    evaluated_run = run_real(real_settings)
else:
    print("真实模型评测：未开启；以下教学数字只来自手工夹具。")

真实模型评测：未开启；以下教学数字只来自手工夹具。


## 11. 原始记录可以脱离GPU重新评分

每次真实运行创建独立时间戳目录，避免把新失败与旧结果混在一起：

```
manifest.json      # 实验设置、环境、完成状态
 dataset.json      # 本次选题及参考答案
 requests.json     # 给模型的公共请求，不含expected
 inspection.json   # 配套分词器、模板与编号
 raw.jsonl         # 每题原始响应，逐条落盘
 report.json       # 逐题评分、分类汇总与主指标
```

`--score` 根据原始响应重算，检查数据/协议/响应指纹，不相信手工编辑过的summary数字。`--compare` 要求相同协议和范围；会拒绝夹具与真实模型混比。

报告含输入输出明文，分享前检查隐私。本课单进程运行，没有自动恢复中断或重复调用后选择最佳回答的功能。

In [21]:
def rescore_directory(directory: str | Path) -> dict:
    path = Path(directory)
    manifest = json.loads((path / "manifest.json").read_text(encoding="utf-8"))
    items = json.loads((path / "dataset.json").read_text(encoding="utf-8"))
    if manifest.get("protocol") != protocol_for(items):
        raise ValueError("数据/提示/评分器版本已变，拒绝悄悄复用旧协议")
    if "condition" not in manifest:
        raise ValueError("模型尚未加载成功或只是inspect，没有真实回答可复评分")
    raw_file = path / "raw.jsonl"
    if not raw_file.exists():
        raise ValueError("没有原始响应文件")
    rows = [json.loads(line) for line in raw_file.read_text(encoding="utf-8").splitlines() if line.strip()]
    saved_hash = manifest.get("raw_records_sha256")
    if saved_hash is not None and saved_hash != fingerprint(rows):
        raise ValueError("原始响应已改变，不能把编辑后的结果当成原运行重新评分")
    report = build_report(items, rows, kind="real_model", condition=manifest["condition"],
                          scope=manifest["scope"], complete=manifest["status"] == "completed")
    json_write(path / "rescored.json", report)
    return report


def load_report(path: str | Path) -> dict:
    path = Path(path)
    if path.is_dir():
        path = path / "report.json"
    report = json.loads(path.read_text(encoding="utf-8"))
    verify_report(report)
    return report

In [22]:
# 只演示本地报告完整性，不调用任何模型。
with tempfile.TemporaryDirectory() as tmp:
    file = Path(tmp) / "fixture_report.json"
    json_write(file, fixture_a)
    reloaded = load_report(file)
    print("原始响应重新评分完全一致：", reloaded["metrics"] == fixture_a["metrics"])
    tampered = copy.deepcopy(reloaded)
    tampered["metrics"]["success"] = 24
    try:
        verify_report(tampered)
    except ValueError as exc:
        print("篡改汇总会被拒绝：", exc)

原始响应重新评分完全一致： True
篡改汇总会被拒绝： 保存的评分与原始响应不一致；必须重新评分


## 12. 自动检查与接口替身

以下两块是可复用的测试代码。第一块只在已有PyTorch时验证生成封装：EOS、前缀切片、模式/RNG恢复、异常传播。它使用很小的手工模型替身，不加载Qwen。

第二块验证题集、评分器、固定分母、完整性和配对比较。**测试通过意味着评测程序通过这些检查，不意味着模型回答正确。**

In [23]:
def backend_contract_checks() -> dict:
    """只验证本课generate封装；不用Transformers、不加载权重。"""
    try:
        import torch
        from types import SimpleNamespace
    except ImportError:
        return {"passed": 0, "skipped": "未安装PyTorch，标准库评分器测试不受影响"}
    names = []

    def check(name, condition):
        if not condition:
            raise AssertionError(name)
        names.append(name)

    class ToyTokenizer:
        pad_token_id = 0
        eos_token_id = 7

        def apply_chat_template(self, messages, **kwargs):
            if not kwargs.get("add_generation_prompt") or kwargs.get("enable_thinking") is not False:
                raise AssertionError("模板参数错误")
            return "测试提示：" + messages[-1]["content"]

        def __call__(self, text, **kwargs):
            if kwargs.get("add_special_tokens") is not False or kwargs.get("truncation") is not False:
                raise AssertionError("编码参数错误")
            return {"input_ids": torch.tensor([[11, 12, 13]]), "attention_mask": torch.ones((1, 3), dtype=torch.long)}

        def decode(self, ids, **kwargs):
            table = {4: "4", 5: "5", 7: "<eosA>", 8: "<eosB>"}
            return "".join(table.get(i, "?") for i in ids)

    class ToyConfig:
        def __init__(self, **kwargs):
            self.options = kwargs
        def to_dict(self):
            return copy.deepcopy(self.options)

    class ToyModel(torch.nn.Module):
        def __init__(self):
            super().__init__()
            self.anchor = torch.nn.Parameter(torch.tensor([1.0]))
            self.dropout = torch.nn.Dropout(0.2)
            self.config = SimpleNamespace(max_position_embeddings=128)
            self.generation_config = SimpleNamespace(eos_token_id=[7, 8])
            self.fail = False
            self.seen = None
        def generate(self, input_ids, attention_mask, generation_config):
            self.seen = generation_config.to_dict()
            if self.training or torch.is_grad_enabled():
                raise AssertionError("生成时没有关闭训练/梯度")
            torch.rand(3)  # 刻意消耗RNG，检查调用后恢复。
            if self.fail:
                raise RuntimeError("手工注入失败")
            tail = torch.tensor([[4, 5, 8]], dtype=torch.long, device=input_ids.device)
            return SimpleNamespace(sequences=torch.cat((input_ids, tail), dim=1))

    model, tokenizer = ToyModel(), ToyTokenizer()
    model.train(); model.dropout.eval()  # 检查混合模式也能逐模块恢复。
    model.anchor.grad = torch.tensor([2.0])
    modes = [m.training for m in model.modules()]
    weights, grads, state = model.anchor.detach().clone(), model.anchor.grad.clone(), torch.get_rng_state().clone()
    request = public_request(select_items(make_suite(), "eval")[0])
    out = generate_one(model, tokenizer, request, RunSettings(), torch.device("cpu"), ToyConfig)
    check("封装只返回新增答案", out["raw_text"] == "45" and out["generated_token_ids"] == [4, 5, 8])
    check("封装使用第二EOS", out["stop_reason"] == "eos" and out["raw_with_eos"] == "45<eosB>")
    check("封装保留前缀宽度", out["input_tokens"] == 3 and out["input_ids"] == [11, 12, 13])
    check("封装所有EOS传入", model.seen["eos_token_id"] == [7, 8])
    check("封装恢复模型模式", [m.training for m in model.modules()] == modes)
    check("封装不改参数梯度", torch.equal(weights, model.anchor) and torch.equal(grads, model.anchor.grad))
    check("封装恢复CPU随机状态", torch.equal(state, torch.get_rng_state()))
    check("封装记录墙钟", out["generate_seconds"] >= 0)
    check("封装评分可接通", grade_one(select_items(make_suite(), "eval")[0], out)["success"])
    model.fail = True
    try:
        generate_one(model, tokenizer, request, RunSettings(), torch.device("cpu"), ToyConfig)
    except RuntimeError:
        pass
    else:
        raise AssertionError("注入错误未传播")
    check("失败后仍恢复模式", [m.training for m in model.modules()] == modes)
    check("失败后仍恢复RNG", torch.equal(state, torch.get_rng_state()))
    return {"passed": len(names), "names": names, "scope": "手工替身的接口测试，不是Qwen模型实测"}

In [24]:
def run_checks(verbose=True) -> dict:
    passed = []

    def check(name, condition):
        if not condition:
            raise AssertionError(name)
        passed.append(name)

    def rejects(name, fn, error=ValueError):
        try:
            fn()
        except error:
            passed.append(name)
        else:
            raise AssertionError(name + "：应该拒绝但没有拒绝")

    suite = make_suite()
    items = select_items(suite, "eval")
    dev = select_items(suite, "dev")
    check("题量", len(suite) == 28 and len(items) == 24 and len(dev) == 4)
    check("每类6题", Counter(x["task"] for x in items) == {t: 6 for t in TASKS})
    rejects("重复ID", lambda: validate_suite(items + [items[0]]))
    bad = copy.deepcopy(items); bad[0]["expected"] = True
    rejects("bool不是整数参考答案", lambda: validate_suite(bad))
    bad = copy.deepcopy(items); bad[0]["task"] = "unknown"
    rejects("拒绝未知任务", lambda: validate_suite(bad))
    bad = copy.deepcopy(items); bad[1]["prompt"] = bad[0]["prompt"]
    rejects("重复题干", lambda: validate_suite(bad))
    rejects("空题集", lambda: score_records([], []))
    rejects("limit布尔值", lambda: select_items(suite, limit=True))
    check("题集函数无共享可变状态", make_suite() == suite and make_suite() is not suite)
    request = public_request(items[0])
    changed = copy.deepcopy(items[0]); changed["expected"] = 999999
    check("答案不进入prompt", request == public_request(changed) and set(request) == {"id", "messages"})
    check("prompt包含明确格式", "ASCII" in request["messages"][1]["content"])
    protocol = protocol_for(items)
    altered = copy.deepcopy(items); altered[0]["expected"] = 4444
    check("答案变化改变数据指纹", protocol != protocol_for(altered))
    check("相同协议稳定", protocol == protocol_for(copy.deepcopy(items)))
    check("JSON键顺序不影响指纹", fingerprint({"x": 1, "y": 2}) == fingerprint({"y": 2, "x": 1}))

    for text, expected in [("45", 45), (" -4\n", -4), ("0", 0), ("156", 156)]:
        value, err = parse_answer("arithmetic", text)
        check("整数正确解析:" + repr(text), err is None and value == expected)
    for text in ["答案是45", "145", "45 or 44", "45.0", "+45", "045", "４５", "-0", "", "45\n解释"]:
        value, err = parse_answer("arithmetic", text)
        check("不宽松提取:" + repr(text), err is not None or value != 45)
    check("标签trim", parse_answer("routing", " PAYMENT\n") == ("PAYMENT", None))
    check("标签不改变大小写", parse_answer("routing", "payment")[1] == "label_format")
    check("标签额外解释失败", parse_answer("routing", "PAYMENT because...")[1] is not None)
    check("正确JSON", parse_answer("extraction", '{"quantity":3,"item":"红茶"}')[1] is None)
    for text in ['```json\n{"item":"红茶","quantity":3}\n```',
                 '{"item":"红茶","quantity":3,}',
                 '{"item":"红茶","item":"别的","quantity":3}',
                 '{"item":"红茶","quantity":NaN}',
                 '{"item":"红茶","quantity":Infinity}',
                 '{"item":"红茶","quantity":3} 后缀']:
        check("拒绝非严格JSON:" + text[:20], parse_answer("extraction", text)[1] == "json_syntax_or_duplicate_key")
    for quantity in [True, 3.0, "3", None, -1]:
        text = json.dumps({"item": "红茶", "quantity": quantity})
        check("JSON数量类型:" + repr(quantity), parse_answer("extraction", text)[1] == "json_schema")
    check("多余JSON字段", parse_answer("extraction", '{"item":"红茶","quantity":3,"note":"ok"}')[1] == "json_schema")
    check("缺少JSON字段", parse_answer("extraction", '{"item":"红茶"}')[1] == "json_schema")
    check("顶层JSON列表", parse_answer("extraction", '[{"item":"红茶","quantity":3}]')[1] == "json_schema")
    check("bool类型严格比较", not same_typed_value(True, 1))
    check("float类型严格比较", not same_typed_value(1.0, 1))
    check("JSON顺序不重要", same_typed_value({"quantity": 3, "item": "红茶"}, {"item": "红茶", "quantity": 3}))

    item = items[0]
    good = {"id": item["id"], "status": "ok", "raw_text": "45", "stop_reason": "eos"}
    check("正确且EOS成功", grade_one(item, good)["success"])
    length = {**good, "stop_reason": "length"}
    score = grade_one(item, length)
    check("正确但被截断分开计数", score["answer_correct"] and not score["success"])
    check("数字包含不算对", not grade_one(item, {**good, "raw_text": "145"})["answer_correct"])
    check("缺失计失败", grade_one(item, None)["reason"] == "missing")
    error_row = {"id": item["id"], "status": "error", "error": {"type": "X", "message": "x"}}
    check("错误不计生成", not grade_one(item, error_row)["generated"])
    rejects("ID错位", lambda: grade_one(item, {**good, "id": "wrong"}))
    rejects("重复响应拒绝", lambda: score_records(items, [good, good]))
    rejects("额外响应拒绝", lambda: score_records(items, [{**good, "id": "not-in-suite"}]))
    rejects("未知status拒绝", lambda: score_records(items, [{**good, "status": "retry_best"}]))
    rejects("未知停止原因拒绝", lambda: score_records(items, [{**good, "stop_reason": "timeout"}]))
    empty = score_records(items, [])["metrics"]
    check("全部缺失分母仍24", empty["n"] == 24 and empty["micro_success_rate"] == 0)
    check("无时间数据不是零", empty["median_generate_seconds"] is None)
    fixture = fixture_records(items)
    report = build_report(items, fixture, kind="hand_authored_fixture", condition={"fixture": "A"}, scope="full_eval")
    m = report["metrics"]
    check("夹具严格成功15", m["success"] == 15 and m["n"] == 24)
    check("正确答案16与成功分开", m["answer_correct"] == 16)
    check("缺失和运行错误保留", m["received"] == 23 and m["generated"] == 22)
    check("主分母不是成功生成数量", m["micro_success_rate"] == 15 / 24)
    check("重评一致", verify_report(report)["metrics"] == m)
    graded = []
    for i in range(9):
        graded.append({"id": str(i), "task": "a", "present": True, "generated": True,
                       "format_valid": True, "answer_correct": True, "success": True, "reason": "success"})
    graded.append({"id": "9", "task": "b", "present": True, "generated": True,
                   "format_valid": True, "answer_correct": False, "success": False, "reason": "wrong_value"})
    agg = summary_metrics(graded, [])
    check("micro宏平均不同", agg["micro_success_rate"] == 0.9 and agg["macro_success_rate"] == 0.5)
    alt = copy.deepcopy(fixture)
    for row in alt:
        if row["id"] == "eval-arithmetic-05": row["raw_text"] = "23"
        if row["id"] == "eval-code_trace-01": row["raw_text"] = "999"
    other = build_report(items, alt, kind="hand_authored_fixture", condition={"fixture": "B"}, scope="full_eval")
    pair = paired_compare(report, other)
    check("同分不等于逐题相同", pair["delta_percentage_points"] == 0 and len(pair["fixed_ids"]) == len(pair["regressed_ids"]) == 1)
    bad_report = copy.deepcopy(report); bad_report["metrics"]["success"] = 24
    rejects("拒绝篡改分数", lambda: verify_report(bad_report))
    bad_report = copy.deepcopy(report); bad_report["raw_records"][0]["raw_text"] = "999"
    rejects("拒绝响应指纹变化", lambda: verify_report(bad_report))
    bad_report = copy.deepcopy(report); bad_report["items"][0]["expected"] = 999
    rejects("拒绝改参考答案", lambda: verify_report(bad_report))
    actual = build_report(items, fixture, kind="real_model", condition={}, scope="full_eval")
    rejects("不比较夹具与真实", lambda: paired_compare(report, actual))
    partial = copy.deepcopy(report); partial["complete"] = False
    rejects("不比较未完成运行", lambda: paired_compare(report, partial))
    smaller = build_report(items[:6], fixture[:6], kind="hand_authored_fixture", condition={}, scope="smoke_subset")
    rejects("不比较不同子集", lambda: paired_compare(report, smaller))
    check("seed不依赖顺序", stable_item_seed(29, "x") == stable_item_seed(29, "x"))
    check("题目seed不同", stable_item_seed(29, "x") != stable_item_seed(29, "y"))
    rejects("seed拒绝bool", lambda: stable_item_seed(True, "x"))
    opts = decode_options(RunSettings(), [2, 3], 0)
    check("greedy不继承采样参数", opts["do_sample"] is False and "temperature" not in opts and opts["eos_token_id"] == [2, 3])
    sampled = decode_options(RunSettings(decode="sample"), [2], 0)
    check("sample参数明确", sampled["do_sample"] and sampled["temperature"] == 0.7)
    rejects("非法最大长度", lambda: decode_options(RunSettings(max_new=0), [2], 0))
    rejects("非法EOS", lambda: decode_options(RunSettings(), [True], 0))
    parts = split_generated_ids([8, 9, 4, 3], [8, 9], [2, 3], 4)
    check("第二EOS有效", parts == {"visible_ids": [4], "generated_ids": [4, 3], "stop_reason": "eos"})
    parts = split_generated_ids([8, 9, 4, 5], [8, 9], [2], 2)
    check("达到预算标length", parts["stop_reason"] == "length")
    check("空输出不冒充EOS", split_generated_ids([8], [8], [2], 4)["stop_reason"] == "other")
    rejects("不接受不同前缀", lambda: split_generated_ids([8, 9], [1], [2], 3))
    rejects("不接受超预算", lambda: split_generated_ids([8, 9, 4], [8], [2], 1))
    with tempfile.TemporaryDirectory() as tmp:
        path = Path(tmp) / "report.json"
        json_write(path, report)
        check("报告JSON保存重载", load_report(path) == report)
    backend = backend_contract_checks()
    result = {"passed": len(passed), "names": passed, "optional_torch_backend": backend,
              "scope": "标准库离线评分/协议检查，不是Qwen权重推理检查"}
    if verbose:
        print(f"标准库固定检查通过：{len(passed)} 项。")
        print(f"PyTorch封装替身检查：{backend['passed']} 项；不是Qwen权重实测。")
    return result

In [25]:
check_report = run_checks()
print("标准库检查：", check_report["passed"])
print("可选PyTorch接口检查：", check_report["optional_torch_backend"]["passed"])

标准库固定检查通过：87 项。
PyTorch封装替身检查：11 项；不是Qwen权重实测。
标准库检查： 87
可选PyTorch接口检查： 11


## 13. 独立练习A：严格JSON字段评分

实现 `grade_json_ex(text, expected)`，只返回bool。

要求：允许键顺序和JSON空白变化；拒绝重复键、围栏、多余字段、字符串数量、布尔数量、浮点数量和错误数值。可以复用 `strict_json_loads()`，但不要调用课程的 `parse_answer()` / `grade_one()`。

只有你完成后才把开关改为True。未开启时不算通过。

In [26]:
def grade_json_ex(text, expected):
    raise NotImplementedError("在这里完成严格JSON字段评分")

RUN_EXERCISE_A = False

def check_exercise_a(fn):
    gold = {"item":"红茶", "quantity":3}
    cases = [
        (' {"quantity":3,"item":"红茶"} ', True),
        ('{"item":"红茶","quantity":3.0}', False),
        ('{"item":"红茶","quantity":true}', False),
        ('{"item":"红茶","quantity":"3"}', False),
        ('{"item":"红茶","quantity":4}', False),
        ('{"item":"错","item":"红茶","quantity":3}', False),
        ('{"item":"红茶","quantity":3,"note":"ok"}', False),
        ('```json\n{"item":"红茶","quantity":3}\n```', False),
        ('[]', False), ('', False),
    ]
    for text, expected in cases:
        assert fn(text, gold) is expected, repr(text)
    assert fn('{"item":"a","quantity":true}', {"item":"a","quantity":1}) is False
    assert fn('{"item":"a","quantity":0}', {"item":"a","quantity":0}) is True
    return len(cases) + 2

if RUN_EXERCISE_A:
    print("练习A通过：", check_exercise_a(grade_json_ex), "项")
else:
    print("练习A待完成")

练习A待完成


## 14. 独立练习B：保留分母的聚合

输入是逐题 `graded` 列表，包含缺失与运行失败。实现 `aggregate_ex(graded)` 返回：

```python
{"n": ..., "success": ..., "micro": ..., "macro": ...}
```

不得过滤未生成的题；空列表应拒绝。不要调用 `summary_metrics()`。本练习直接考察分母，而不是字符串解析。

In [27]:
def aggregate_ex(graded):
    raise NotImplementedError("固定总题数，并按任务计算宏平均")

RUN_EXERCISE_B = False

def check_exercise_b(fn):
    got = fn(fixture_a["graded"])
    assert got == {"n":24,"success":15,"micro":15/24,"macro":15/24}
    mixed = [{"task":"a","success":True} for _ in range(9)]
    mixed += [{"task":"b","success":False}]
    assert fn(mixed) == {"n":10,"success":9,"micro":0.9,"macro":0.5}
    assert fn([{"task":"a","success":False}])["micro"] == 0
    try:
        fn([])
    except ValueError:
        pass
    else:
        raise AssertionError("空题集必须拒绝")
    return 4

if RUN_EXERCISE_B:
    print("练习B通过：", check_exercise_b(aggregate_ex), "项")
else:
    print("练习B待完成")

练习B待完成


## 15. 独立练习C：按ID找进步和退步

实现 `paired_ids_ex(before, after)`，输入两个字典列表，各元素包含 `id` 与布尔 `success`。

返回 `{"fixed_ids": [...], "regressed_ids": [...]}`，输出排序。两边ID集合必须一致，且各自不能重复。改变列表行顺序不应改变结果。

完整实验还要检查协议指纹；本练习聚焦最小的ID配对逻辑，不调用 `paired_compare()`。

In [28]:
def paired_ids_ex(before, after):
    raise NotImplementedError("按ID对齐，不按列表行号对齐")

RUN_EXERCISE_C = False

def check_exercise_c(fn):
    a = [{"id":"x","success":False},{"id":"y","success":True}]
    b = [{"id":"y","success":False},{"id":"x","success":True}]
    expected = {"fixed_ids":["x"],"regressed_ids":["y"]}
    assert fn(a,b) == expected
    assert fn(a,list(reversed(b))) == expected
    assert fn(a,a) == {"fixed_ids":[],"regressed_ids":[]}
    for bad in [b[:1], b+[b[0]]]:
        try:
            fn(a,bad)
        except ValueError:
            pass
        else:
            raise AssertionError("ID缺失/重复必须拒绝")
    return 5

if RUN_EXERCISE_C:
    print("练习C通过：", check_exercise_c(paired_ids_ex), "项")
else:
    print("练习C待完成")

练习C待完成


---
## 16. 参考答案（先独立完成练习）

下面使用不同函数名，不覆盖你的实现。能运行参考答案，不等于已经能独立解释和实现。

In [29]:
def grade_json_reference(text, expected):
    try:
        value = strict_json_loads(text)
    except (ValueError, RecursionError):
        return False
    if type(value) is not dict or set(value) != {"item","quantity"}:
        return False
    if type(value["item"]) is not str or type(value["quantity"]) is not int:
        return False
    if value["quantity"] < 0:
        return False
    return value["item"] == expected["item"] and value["quantity"] == expected["quantity"]

print("参考A检查数：", check_exercise_a(grade_json_reference))

参考A检查数： 12


In [30]:
def aggregate_reference(graded):
    if not graded:
        raise ValueError("空题集")
    total = len(graded)
    successes = sum(row["success"] for row in graded)
    groups = {}
    for row in graded:
        group = groups.setdefault(row["task"], {"n":0,"success":0})
        group["n"] += 1
        group["success"] += int(row["success"])
    macro = sum(g["success"] / g["n"] for g in groups.values()) / len(groups)
    return {"n":total,"success":successes,"micro":successes/total,"macro":macro}

print("参考B检查数：", check_exercise_b(aggregate_reference))

参考B检查数： 4


In [31]:
def paired_ids_reference(before, after):
    def index(rows):
        result = {}
        for row in rows:
            if row["id"] in result:
                raise ValueError("重复ID")
            if type(row["success"]) is not bool:
                raise ValueError("success必须为bool")
            result[row["id"]] = row["success"]
        return result
    a,b = index(before),index(after)
    if a.keys() != b.keys():
        raise ValueError("ID集合不一致")
    return {"fixed_ids":sorted(k for k in a if not a[k] and b[k]),
            "regressed_ids":sorted(k for k in a if a[k] and not b[k])}

print("参考C检查数：", check_exercise_c(paired_ids_reference))

参考C检查数： 5


## 17. 脚本命令与输出核对

独立脚本默认是**离线夹具演示**，不是自动下载模型：

```bash
python lesson_29_fixed_tasks_and_automatic_evaluation.py
python lesson_29_fixed_tasks_and_automatic_evaluation.py --checks-only
```

只检查分词器；再运行4道开发题：

```bash
python lesson_29_fixed_tasks_and_automatic_evaluation.py --inspect --split dev
python lesson_29_fixed_tasks_and_automatic_evaluation.py --run --split dev --device cuda --dtype float16
```

冻结配置后运行完整24题：

```bash
python lesson_29_fixed_tasks_and_automatic_evaluation.py --run --split eval --device cuda --dtype float16
```

没有可用CUDA就明确用 `--device cpu`。缓存完整后追加 `--local-files-only`。只跑前几题可用 `--limit 2`，但报告会标为 `smoke_subset`。

离线复评分与配对比较：

```bash
python lesson_29_fixed_tasks_and_automatic_evaluation.py --score lesson_29_outputs/real_实际时间戳
python lesson_29_fixed_tasks_and_automatic_evaluation.py --compare run_A/report.json run_B/report.json
```

参数里的路径请替换为实际生成目录。若修改提示或评分规则，旧协议不能直接参与严格比较；应当保存新版本，并公平地重新运行/重新评分两边。

## 18. 本课验收

你应该能独立回答：

1. 24题里2题运行失败，剩下的题全对，为什么不能只报告100%？
2. `{"quantity":true}` 为什么不符合整数数量字段？
3. 答案正确但没有EOS结束，为什么本课保留两个不同指标？
4. 两次都是15/24，逐题报告还能告诉我们什么？
5. 改了提示、dtype、生成长度后，哪一项才是你真正想测的变量？

**实际模型最终验收**：完成dev4题并检查原始输出；冻结配置后跑eval24题；从原始JSONL重新评分得到同样结果。当前讲义没有预填Qwen成绩，没有把离线测试次数当作模型得分。

下一课：**第30课——权重量化原理：INT8、INT4、缩放因子与误差**。先理解数值映射，再利用本课基线验证后续量化模型的输出变化。

## 来源与验证边界

- [Qwen3-0.6B 固定版本模型卡](https://huggingface.co/Qwen/Qwen3-0.6B/blob/c1899de289a04d12100db370d81485cdf75e47ca/README.md)：配套模板、thinking与非thinking设置。
- [Transformers GenerationConfig](https://huggingface.co/docs/transformers/v4.57.3/en/main_classes/text_generation)：生成预算、采样、停止条件。
- [Transformers Chat Template](https://huggingface.co/docs/transformers/chat_templating)：角色消息与特殊token。
- [Python JSON](https://docs.python.org/3/library/json.html)：重复键、非标准常量、解码钩子。
- [LM Evaluation Harness task guide](https://github.com/EleutherAI/lm-evaluation-harness/blob/main/docs/task_guide.md)：输入、参考答案、评分规则与任务配置。
- [PyTorch reproducibility](https://docs.pytorch.org/docs/stable/notes/randomness.html)：随机数与跨环境复现边界。

题集、评分协议与手工演示是本课自编设计，不冒充官方benchmark。Notebook的主实现与独立脚本来自同一份源码；真实模型部分需要在本机开启，当前制作环境不具备Transformers/权重且外网下载不可用。